In [ ]:
!pip install -q transformers datasets evaluate torch scikit-learn accelerate
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
device_name = tf.test.gpu_device_name()
if device_name != '/device:GPU:0':
  raise SystemError('GPU device not found')
print('Found GPU at: {}'.format(device_name))

Found GPU at: /device:GPU:0


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
import matplotlib.pyplot as plt
from transformers import pipeline
from sklearn.metrics import accuracy_score
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
# Load and Stratify Data
file_path = "/gdrive/MyDrive/OrchidPharmed/hospital_with_neutral_labels_new.csv"  # <--- Ensure this matches your uploaded file
df = pd.read_csv(file_path)

X = df['Feedback'].astype(str).tolist()
y = df['sentiment_label_3class'].tolist()

# SPLIT 1: Hold out 10% for FINAL TESTING
# Stratify ensures the test set represents all classes equally
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

# Validation Dataset (for validating model and save the best model)
val_size = 0.10
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size, random_state=42, stratify=y_temp
)

print(f"Data Sizes -> Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Create Hugging Face Datasets
dataset = DatasetDict({
    'train': Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train})),
    'validation': Dataset.from_pandas(pd.DataFrame({'text': X_val, 'label': y_val})),
    'test': Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))
})

Data Sizes -> Train: 806, Val: 90, Test: 100


In [ ]:
# Visualizing Frequency of Classes
label_counts = df["final_sentiment"].value_counts(ascending=True)
label_counts.plot.barh()
plt.title("Frequency of Classes")
plt.show()
print(label_counts)

# Visualizing the number of words of each comment
df['word_count'] = df['Feedback'].str.split().apply(len)
df.boxplot('word_count', by='final_sentiment')

X = df['Feedback'].astype(str).tolist()
y = df['sentiment_label_3class'].tolist()


In [ ]:
# Compute Class Weights (The Imbalance Fix)
# We only calculate weights based on Training data to avoid data leakage
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"Class Weights (Handling Imbalance): {class_weights}")

Class Weights (Handling Imbalance): [1.09214092 0.513703   7.26126126]


In [ ]:
# Tokenization
model_name = "distilbert-base-uncased" # Changed from "distilbert-finetuned-emotion" to a valid model name
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=False)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/806 [00:00<?, ? examples/s]

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# Custom Trainer for Weighted Loss
# Standard Trainer doesn't accept weights easily. We override it.
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
# Model & Metrics
id2label = {0: "Negative", 1: "Positive", 2: "Neutral"}
label2id = {"Negative": 0, "Positive": 1, "Neutral": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label, label2id=label2id
).to(device)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, predictions, average="macro")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1_macro": f1}

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Training Arguments with Early Stopping
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=30,             # High number, Early Stopping will cut it
    weight_decay=0.01,
    eval_strategy="epoch",     # Evaluate on VAL set every epoch
    save_strategy="epoch",           # Save checkpoint every epoch
    load_best_model_at_end=True,     # Load best model based on VAL metric
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_dir='./logs',
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # <--- Validation used here
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)


/tmp/ipython-input-20600971.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(


In [ ]:
# Train, Evaluate, and Save
print("Starting Training...")
trainer.train()

print("\n--- FINAL TEST SET EVALUATION ---")
# Now we use the TEST set, which the model has never seen before
test_results = trainer.predict(tokenized_datasets["test"])
test_preds = np.argmax(test_results.predictions, axis=1)

print(f"Final Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Final Test F1 (Macro): {f1_score(y_test, test_preds, average='macro'):.4f}")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, test_preds, target_names=["Negative", "Positive", "Neutral"]))

# Save
best_model_path = "/gdrive/MyDrive/OrchidPharmed/best_sentiment_model_testsplit"
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)
print(f"Model saved to {best_model_path}")

Starting Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,0.784499,0.888889,0.682540
2,No log,0.553360,0.877778,0.744035
3,No log,0.442109,0.877778,0.749658
4,No log,0.479617,0.866667,0.672639
5,No log,0.674809,0.888889,0.703065
6,No log,0.641033,0.877778,0.695655
7,No log,0.807887,0.888889,0.703065


Epoch,Training Loss,Validation Loss



--- FINAL TEST SET EVALUATION ---


Final Test Accuracy: 0.9100
Final Test F1 (Macro): 0.8020

Classification Report (Test Set):
              precision    recall  f1-score   support

    Negative       0.96      0.80      0.87        30
    Positive       0.95      0.97      0.96        65
     Neutral       0.44      0.80      0.57         5

    accuracy                           0.91       100
   macro avg       0.79      0.86      0.80       100
weighted avg       0.93      0.91      0.92       100

Model saved to /gdrive/MyDrive/OrchidPharmed/best_sentiment_model_testsplit


In [ ]:
# 1. Load the saved model and tokenizer
# We point to the folder where we saved the "best" model
model_path = "/gdrive/MyDrive/OrchidPharmed/best_sentiment_model_testsplit"

classifier = pipeline(
    "text-classification",
    model=model_path,
    tokenizer=model_path,
    device=0 if torch.cuda.is_available() else -1  # Use GPU if available
)

# 2. Define the text you want to test
text_to_test = "there are good nurses but the environment was not clean"

# 3. Get Prediction
result = classifier(text_to_test)

# 4. Print Result
print(f"Text: '{text_to_test}'")
print(f"Predicted Label: {result[0]['label']}")
print(f"Confidence Score: {result[0]['score']:.4f}")

Device set to use cuda:0


Text: 'there are good nurses but the environment was not clean'
Predicted Label: Neutral
Confidence Score: 0.6149
